# ① 전처리 — 문서 파싱 · 청크 분할
담당: 전처리 · 코드: `rag/preprocessing/parsing.py`, `rag/preprocessing/chunking.py`

`config.yaml` 의 `paths.input`(ZIP 또는 폴더)에 있는 PDF · HWP · HWPX 를 읽어 본문 · 표 · (선택) 그림 설명을 뽑고, 청크로 나눕니다.

| 저장 파일 | 읽는 노트북 |
|---|---|
| `artifacts/chunks.jsonl` (청크) | 02 검색, 03 검증 |
| `artifacts/docs.jsonl` (문서 목록 — 발주기관 라우팅용) | 02 검색 |

## 설정
`OCR = True` 로 바꾸면 문서 속 그림을 OpenAI 비전 모델(`gpt-5-mini`, `.env` 의 `OPENAI_API_KEY`)로 글로 바꿔 청크에 넣습니다 (유료).
로고 · 장식처럼 여러 쪽에 반복되는 그림(`config.yaml` 의 `max_repeat_image_pages` 쪽 이상)은 청크에 넣지 않습니다.
`False` 면 그림은 API 로 보내지 않습니다.

In [ ]:
OCR = False

In [ ]:
from rag.config import ARTIFACTS, ROOT, load_config
from rag.preprocessing.chunking import chunk_documents, chunk_summary, save_chunks
from rag.preprocessing.parsing import collect_inputs, parse_documents, parsing_summary

cfg = load_config()

## 1. 입력 파일 모으기
ZIP(안의 ZIP 포함)을 `artifacts/raw/` 에 풀고 PDF · HWP · HWPX 목록을 만듭니다.

In [ ]:
files = collect_inputs(ROOT / cfg["paths"]["input"], ARTIFACTS / "raw")
len(files)

## 2. 파싱
문서마다 쪽 · 표 · 그림 수와 표지 감지 결과를 봅니다.

In [ ]:
docs = parse_documents(files, cfg["parsing"], ocr=OCR)
parsing_summary(docs)

## 3. 청크 분할 · 저장

In [ ]:
chunks = chunk_documents(docs, cfg["chunking"])
save_chunks(docs, chunks, ARTIFACTS)
chunk_summary(chunks)